# Làm sạch và Chuẩn bị dữ liệu – Dự đoán phí vận chuyển Olist

Nhóm 8, lớp AI2107, môn ADY201M, FALL26

---

## 0. Thiết lập

**Mục đích:** nạp thư viện và cố định seed để kết quả chạy lại giống nhau.

**Hàm step():** mọi thao tác loại dòng khỏi bảng chính df đều đi qua hàm này. Hàm ghi lại số dòng trước và sau mỗi lần lọc, làm cơ sở lập bảng audit trail ở cuối bài.

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
pd.set_option('display.max_columns', None)

audit_rows = []

def step(frame, keep, section, rule):
    truoc = len(frame)
    out = frame[keep].copy()
    audit_rows.append({
        'muc': section, 'quy_tac': rule,
        'truoc': truoc, 'sau': len(out), 'loai': truoc - len(out),
    })
    print(f'{section:<12} {rule:<50} {truoc:>8,} -> {len(out):>8,} (loai {truoc-len(out):,})')
    return out

Kết nối Google Drive để đọc và ghi dữ liệu.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


---

## 3. Làm sạch dữ liệu

### 3.1 Đánh giá chất lượng dữ liệu

**Mục đích:** đánh giá chất lượng dữ liệu gốc trước khi xử lý.

#### 3.1.1 Kích thước dữ liệu

**Câu hỏi:** mỗi bảng sau khi đọc có bao nhiêu dòng và bao nhiêu cột?

**Cách làm:**

- Đọc 7 tệp CSV.
- In kích thước (số dòng, số cột) của từng bảng.

In [ ]:
raw = '/content/drive/MyDrive/Colab Notebooks/'

orders    = pd.read_csv(raw + 'olist_orders_dataset.csv')
items     = pd.read_csv(raw + 'olist_order_items_dataset.csv')
products  = pd.read_csv(raw + 'olist_products_dataset.csv')
customers = pd.read_csv(raw + 'olist_customers_dataset.csv')
geo       = pd.read_csv(raw + 'olist_geolocation_dataset.csv')
sellers   = pd.read_csv(raw + 'olist_sellers_dataset.csv')
cat_map   = pd.read_csv(raw + 'product_category_name_translation.csv')

tables = {
    'orders': orders, 'items': items, 'products': products, 'customers': customers,
    'geo': geo, 'sellers': sellers, 'cat_map': cat_map,
}

for name, t in tables.items():
    print(f'{name:<10}: {t.shape}')

orders    : (99441, 8)
items     : (112650, 7)
products  : (32951, 9)
customers : (99441, 5)
geo       : (1000163, 5)
sellers   : (3095, 4)
cat_map   : (71, 2)


> **Kết luận:** 7 bảng có kích thước như sau.
>
> - orders: 99,441 dòng, 8 cột.
> - items: 112,650 dòng, 7 cột.
> - products: 32,951 dòng, 9 cột.
> - customers: 99,441 dòng, 5 cột.
> - geo: 1,000,163 dòng, 5 cột.
> - sellers: 3,095 dòng, 4 cột.
> - cat_map: 71 dòng, 2 cột.

#### 3.1.2 Kiểu dữ liệu

**Câu hỏi:** kiểu dữ liệu của các cột có phù hợp để phân tích không?

**Cách làm:** in kiểu dữ liệu của từng cột trong 7 bảng; chỉ kiểm tra, không chuyển đổi.

In [ ]:
for name, t in tables.items():
    print('---', name, '---')
    print(t.dtypes)

--- orders ---
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
--- items ---
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object
--- products ---
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object
--- customers ---
customer_id    

> **Kết luận:** các cột số có kiểu số; 6 cột ngày đang ở kiểu object (chuỗi) nên cần chuyển sang datetime trước khi tính toán theo thời gian.
>
> - orders: order_purchase_timestamp, order_approved_at, order_delivered_carrier_date, order_delivered_customer_date, order_estimated_delivery_date.
> - items: shipping_limit_date.
> - Cột order_purchase_timestamp được chuyển ở 3.5.2, vì đây là cột duy nhất cần dùng để lọc thời gian.

#### 3.1.3 Giá trị thiếu

**Câu hỏi:** bảng và cột nào có giá trị thiếu, với số lượng bao nhiêu?

**Cách làm:** đếm giá trị thiếu (NaN) của từng cột; chỉ hiển thị cột có thiếu.

In [ ]:
for name, t in tables.items():
    miss = t.isnull().sum()
    print('---', name, '---')
    if miss.sum() > 0:
        print(miss[miss > 0])
    else:
        print('không thiếu')

--- orders ---
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64
--- items ---
không thiếu
--- products ---
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64
--- customers ---
không thiếu
--- geo ---
không thiếu
--- sellers ---
không thiếu
--- cat_map ---
không thiếu


> **Kết luận:** chỉ 2 trong 7 bảng có giá trị thiếu.
>
> - orders: thiếu ở 3 cột ngày, lần lượt 160, 1,783 và 2,965 dòng.
> - products: thiếu 610 dòng ở 4 cột mô tả, và 2 dòng ở 4 cột cân nặng/kích thước.

#### 3.1.4 Dữ liệu trùng lặp

**Câu hỏi:** khóa của các bảng có duy nhất không; có tồn tại dòng trùng hoàn toàn không?

**Cách làm:**

- Kiểm tra tính duy nhất của khóa ở orders, products, cat_map, customers, sellers.
- Kiểm tra items theo khóa ghép (order_id, order_item_id).
- Đếm dòng trùng hoàn toàn ở cả 7 bảng.

In [ ]:
print('orders.order_id                     :', orders['order_id'].is_unique)
print('products.product_id                 :', products['product_id'].is_unique)
print('cat_map.product_category_name       :', cat_map['product_category_name'].is_unique)
print('customers.customer_id               :', customers['customer_id'].is_unique)
print('sellers.seller_id                   :', sellers['seller_id'].is_unique)
print('items(order_id, order_item_id)      :', not items[['order_id', 'order_item_id']].duplicated().any())
print('geo.geolocation_zip_code_prefix     :', geo['geolocation_zip_code_prefix'].is_unique)
print()
for name,t in tables.items():
    print(f'trùng toàn dòng {name:<10}: {t.duplicated().sum()}')

orders.order_id                     : True
products.product_id                 : True
cat_map.product_category_name       : True
customers.customer_id               : True
sellers.seller_id                   : True
items(order_id, order_item_id)      : True
geo.geolocation_zip_code_prefix     : False

trùng toàn dòng orders    : 0
trùng toàn dòng items     : 0
trùng toàn dòng products  : 0
trùng toàn dòng customers : 0
trùng toàn dòng geo       : 261831
trùng toàn dòng sellers   : 0
trùng toàn dòng cat_map   : 0


> **Kết luận:** 6 bảng có khóa duy nhất; geo không có khóa duy nhất và là bảng duy nhất có dòng trùng hoàn toàn.
>
> - Khóa: orders, products, cat_map, customers, sellers duy nhất theo khóa chính; items duy nhất theo khóa ghép (order_id, order_item_id); geo có geolocation_zip_code_prefix không duy nhất.
> - Dòng trùng hoàn toàn: geo có 261,831 dòng; 6 bảng còn lại có 0 dòng.

#### 3.1.5 Giá trị sai miền và outlier thống kê

**Câu hỏi:** các biến số có giá trị sai miền hoặc outlier thống kê không?

**Phân biệt:** hai loại giá trị bất thường có cách xử lý khác nhau.

- Sai miền: giá trị nằm ngoài khoảng mà biến được phép nhận, nên chắc chắn là lỗi dữ liệu; được loại hoặc sửa ở bước làm sạch.
- Outlier thống kê: giá trị nằm trong khoảng mà biến được phép nhận nhưng lớn hơn hoặc nhỏ hơn bất thường so với đa số giá trị còn lại; chỉ ghi nhận, không loại ở bước làm sạch.

**Cách làm:**

- Thống kê mô tả price, freight_value, product_weight_g.
- Đếm số dòng vi phạm miền giá trị của các biến này và của tọa độ geo, theo khung bao Brazil (vĩ độ từ -34 đến 5.3, kinh độ từ -74 đến -34).
- Đếm số giá trị lớn hơn Q3 + 1.5 × IQR của ba biến đầu.

**Lưu ý:** các thống kê ở bước này chỉ dùng để mô tả, không dùng để biến đổi dữ liệu.

In [ ]:
print(items[['price','freight_value']].describe())
print()
print(products[['product_weight_g']].describe())
print()
print('price<=0        :', (items['price']<=0).sum())
print('freight_value<0 :', (items['freight_value']<0).sum())
print('freight_value=0 :', (items['freight_value']==0).sum())
print('weight_g<=0     :', (products['product_weight_g']<=0).sum())
print()
lat_in=geo['geolocation_lat'].between(-34,5.3)
lng_in=geo['geolocation_lng'].between(-74,-34)
print('geo ngoài Brazil:',(~(lat_in&lng_in)).sum())
print()
q1=items['price'].quantile(0.25)
q3=items['price'].quantile(0.75)
print('price outlier :', (items['price'] > q3 + 1.5*(q3-q1)).sum())
q1=items['freight_value'].quantile(0.25)
q3=items['freight_value'].quantile(0.75)
print('freight_value outlier :', (items['freight_value'] > q3 + 1.5*(q3-q1)).sum())
q1=products['product_weight_g'].quantile(0.25)
q3=products['product_weight_g'].quantile(0.75)
print('weight_g outlier :', (products['product_weight_g'] > q3 + 1.5*(q3-q1)).sum())

               price  freight_value
count  112650.000000  112650.000000
mean      120.653739      19.990320
std       183.633928      15.806405
min         0.850000       0.000000
25%        39.900000      13.080000
50%        74.990000      16.260000
75%       134.900000      21.150000
max      6735.000000     409.680000

       product_weight_g
count      32949.000000
mean        2276.472488
std         4282.038731
min            0.000000
25%          300.000000
50%          700.000000
75%         1900.000000
max        40425.000000

price<=0        : 0
freight_value<0 : 0
freight_value=0 : 383
weight_g<=0     : 4

geo ngoài Brazil: 42

price outlier : 8427
freight_value outlier : 11613
weight_g outlier : 4551


> **Kết luận:** sai miền xuất hiện ở product_weight_g và tọa độ geo; price và freight_value không có giá trị sai miền nhưng phân phối lệch phải.
>
> - price: 0 dòng nhỏ hơn hoặc bằng 0; trung bình 120.65 so với trung vị 74.99, giá trị lớn nhất 6,735.
> - freight_value: 0 dòng âm, 383 dòng bằng 0; trung bình 19.99 so với trung vị 16.26, giá trị lớn nhất 409.68.
> - product_weight_g: 4 dòng bằng 0; giá trị lớn nhất 40,425 so với trung vị 700.
> - geo: 42 điểm tọa độ nằm ngoài khung bao Brazil (vĩ độ -34 đến 5.3, kinh độ -74 đến -34, xấp xỉ các cực của lãnh thổ).
> - Outlier thống kê (lớn hơn Q3 + 1.5 × IQR): price 8,427 dòng, freight_value 11,613 dòng, product_weight_g 4,551 dòng; chỉ ghi nhận, không loại.

#### 3.1.6 Quan hệ giữa các bảng

**Câu hỏi:** khóa ngoại ở bảng con có tồn tại đầy đủ ở bảng cha không?

**Cách làm:**

- Đếm số giá trị khóa ngoại không có trong khóa của bảng cha ở 4 quan hệ: items với products, orders, sellers và orders với customers.
- Đếm số sản phẩm có tên danh mục không có bản dịch trong cat_map.

**Lưu ý:** 610 sản phẩm không có tên danh mục (phát hiện ở 3.1.3) nên không có giá trị để dịch; các sản phẩm này không được tính ở phép đếm cuối.

In [ ]:
print('items.product_id  -> products.product_id          :', (~items['product_id'].isin(products['product_id'])).sum())
print('items.order_id    -> orders.order_id              :',(~items['order_id'].isin(orders['order_id'])).sum())
print('items.seller_id -> sellers.seller_id              :',(~items['seller_id'].isin(sellers['seller_id'])).sum())
print('orders.customer_id-> customers.customer_id        :',(~orders['customer_id'].isin(customers['customer_id'])).sum())
cats=products['product_category_name'].dropna()
print('products.product_category_name -> cat_map.product :', (~cats.isin(cat_map['product_category_name'])).sum())

items.product_id  -> products.product_id          : 0
items.order_id    -> orders.order_id              : 0
items.seller_id -> sellers.seller_id              : 0
orders.customer_id-> customers.customer_id        : 0
products.product_category_name -> cat_map.product : 13


> **Kết luận:** 4 quan hệ khóa ngoại đều toàn vẹn; bảng dịch cat_map chưa bao phủ hết danh mục của products.
>
> - items với products, items với orders, items với sellers, orders với customers: 0 giá trị không có trong bảng cha.
> - products với cat_map: 13 sản phẩm có tên danh mục không có bản dịch.

### 3.2 Xử lý dữ liệu trùng lặp và vá bảng phụ

**Mục đích:** chuẩn bị các bảng phụ trước khi ghép: vá bản dịch còn thiếu của cat_map và đưa geo về một dòng cho mỗi zip_code_prefix, để việc ghép ở 3.3 không nhân dòng và hạn chế giá trị thiếu.

#### 3.2.1 Vá bảng dịch danh mục

**Mục đích:** cat_map thiếu bản dịch của một số danh mục, tương ứng 13 sản phẩm đã phát hiện ở 3.1.6; nếu không vá, các sản phẩm này thiếu tên danh mục tiếng Anh sau khi ghép bảng.

**Cách làm:**

- Xác định các danh mục thiếu bản dịch.
- Dịch tay rồi nối vào cat_map.
- Kiểm tra khóa vẫn duy nhất và không còn danh mục thiếu bản dịch.

**Bước 1:** liệt kê các danh mục có trong products nhưng không có trong cat_map, kèm số sản phẩm của từng danh mục.

In [ ]:
thieu=cats[~cats.isin(cat_map['product_category_name'])]
print(thieu.value_counts())

product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


> **Kết luận:** 2 danh mục không có trong cat_map, gồm 13 sản phẩm.
>
> - portateis_cozinha_e_preparadores_de_alimentos: 10 sản phẩm.
> - pc_gamer: 3 sản phẩm.

**Bước 2:** bổ sung bản dịch của 2 danh mục thiếu vào bản sao cat_map_new; pc_gamer giữ nguyên vì là thuật ngữ quốc tế, portateis_cozinha_e_preparadores_de_alimentos dịch sát nghĩa thành portable_kitchen_and_food_preparers; kiểm tra khóa vẫn duy nhất và không còn danh mục thiếu bản dịch.

In [ ]:
cat_map_new = cat_map.copy()
cat_map_new.loc[len(cat_map_new)] = ['pc_gamer', 'pc_gamer']
cat_map_new.loc[len(cat_map_new)] = ['portateis_cozinha_e_preparadores_de_alimentos', 'portable_kitchen_and_food_preparers']

print('số dòng cat_map     :', len(cat_map))
print('số dòng cat_map_new :', len(cat_map_new))
print('khóa duy nhất       :', cat_map_new['product_category_name'].is_unique)
print('danh mục còn thiếu  :', (~cats.isin(cat_map_new['product_category_name'])).sum())

số dòng cat_map     : 71
số dòng cat_map_new : 73
khóa duy nhất       : True
danh mục còn thiếu  : 0


> **Kết luận:** cat_map_new có 73 dòng, khóa duy nhất, không còn danh mục thiếu bản dịch; các bước sau dùng cat_map_new thay cho cat_map.

#### 3.2.2 Khử trùng geolocation và tính tâm theo zip prefix

**Nguyên nhân:**

- geo là bảng tra cứu tọa độ theo zip_code_prefix khi ghép với customers và sellers, nhưng zip_code_prefix không duy nhất (3.1.4) nên ghép trực tiếp làm nhân dòng bảng chính.
- geo có 42 dòng nằm ngoài khung bao Brazil (3.1.5), làm sai tọa độ tra cứu.

**Cách làm:**

- Loại các dòng ngoài khung bao Brazil.
- Gộp theo zip_code_prefix và lấy trung vị tọa độ, mỗi mã còn một dòng.
- Không loại riêng các dòng trùng hoàn toàn (3.1.4); mỗi zip_code_prefix được gom về một dòng ở Bước 2.
- Tính lại trung vị trên dữ liệu đã bỏ trùng để so sánh độ lệch tọa độ với zip_info (Bước 3).

**Bước 1:** loại các dòng có tọa độ ngoài khung bao Brazil (vĩ độ từ -34 đến 5.3, kinh độ từ -74 đến -34).

In [ ]:
lat_in=geo['geolocation_lat'].between(-34,5.3)
lng_in=geo['geolocation_lng'].between(-74,-34)
geo_in=geo[lat_in&lng_in]
print(len(geo),' -> ',len(geo_in))

1000163  ->  1000121


> **Kết luận:** geo giảm từ 1,000,163 xuống 1,000,121 dòng; 42 dòng ngoài khung bao Brazil đã bị loại, khớp số liệu ở 3.1.5.

**Bước 2:** gộp geo_in theo zip_code_prefix, lấy trung vị của vĩ độ và kinh độ; trung vị ít bị lệch bởi các điểm sai vị trí còn nằm trong khung bao hơn trung bình. Kiểm tra khóa duy nhất của kết quả.

In [ ]:
zip_info=geo_in.groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].median()
zip_info.columns=['lat','lng']
print(len(geo_in), '->', len(zip_info))
print('khóa duy nhất:', zip_info.index.is_unique)

1000121 -> 19010
khóa duy nhất: True


> **Kết luận:** geo_in gộp từ 1,000,121 dòng xuống 19,010 dòng, mỗi zip_code_prefix một dòng; khóa zip_code_prefix của zip_info duy nhất.

**Bước 3:** tính lại trung vị trên dữ liệu đã bỏ trùng hoàn toàn, rồi so với zip_info để xem việc không loại riêng các dòng trùng có làm lệch tọa độ không.

In [ ]:
zip_info2=geo_in.drop_duplicates().groupby('geolocation_zip_code_prefix')[['geolocation_lat','geolocation_lng']].median()
zip_info2.columns=['lat','lng']
lech=(zip_info2 - zip_info).abs()
print('Độ lệch trung bình:')
print(lech.mean())
print('Độ lệch trung vị:')
print(lech.median())
print('Số mã lệch quá 0.01 độ:')
print((lech > 0.01).sum())
print('Số mã lệch quá 0.1 độ:')
print((lech > 0.1).sum())
print('Độ lệch lớn nhất:')
print(lech.max())

Độ lệch trung bình:
lat    0.002701
lng    0.002694
dtype: float64
Độ lệch trung vị:
lat    0.000110
lng    0.000116
dtype: float64
Số mã lệch quá 0.01 độ:
lat    151
lng    155
dtype: int64
Số mã lệch quá 0.1 độ:
lat    21
lng    24
dtype: int64
Độ lệch lớn nhất:
lat     9.369459
lng    10.387792
dtype: float64


> **Kết luận:** tọa độ tính trên dữ liệu đã bỏ trùng lệch rất ít so với zip_info ở hầu hết các mã, nhưng có một số ít mã lệch lớn.
>
> - Độ lệch trung bình: 0.003 độ ở cả vĩ độ và kinh độ; trung vị độ lệch: 0.0001 độ.
> - Số mã lệch quá 0.01 độ: 151 (vĩ độ) và 155 (kinh độ) trong 19,010 mã.
> - Số mã lệch quá 0.1 độ: 21 (vĩ độ) và 24 (kinh độ).
> - Độ lệch lớn nhất: 9.37 độ (vĩ độ) và 10.39 độ (kinh độ).
> - Quyết định: giữ zip_info, không thay bằng zip_info2, vì ít nhất 99.87% số mã (18,986 trên 19,010) lệch không quá 0.1 độ ở mỗi chiều và trung vị độ lệch chỉ 0.0001 độ; 21 đến 24 mã lệch lớn còn lại được ghi nhận.

### 3.3 Ghép các bảng

**Mục đích:** gộp 7 bảng về một bảng, mỗi dòng là một sản phẩm trong một đơn, có đủ freight_value và các biến đầu vào.

**Cách làm:**

- Lấy items làm bảng gốc, nối lần lượt orders, products, cat_map_new, customers, sellers và zip_info bằng left join; zip_info nối hai lần cho tọa độ khách hàng và tọa độ người bán.
- So số dòng trước và sau khi ghép để chắc chắn không nhân hay mất dòng.

**Lưu ý:** việc ghép chỉ thực hiện sau khi đã xử lý các bảng phụ, vì hai lý do:

- Khóa của bảng được nối phải duy nhất để không nhân dòng: các bảng orders, products, customers, sellers, cat_map duy nhất (3.1.4), riêng geo không có khóa nên được gộp thành zip_info có khóa duy nhất (3.2.2).
- Khóa ngoại phải khớp để không phát sinh giá trị thiếu: các khóa ngoại khớp đủ (3.1.6), riêng 13 sản phẩm thiếu bản dịch danh mục đã được vá bằng cat_map_new (3.2.1).

In [ ]:
df=(items
    .merge(orders[['order_id','customer_id','order_status','order_purchase_timestamp']],on='order_id',how='left')
    .merge(products[['product_id','product_category_name','product_weight_g','product_length_cm','product_height_cm','product_width_cm']],on='product_id',how='left')
    .merge(cat_map_new,on='product_category_name',how='left')
    .merge(customers[['customer_id','customer_zip_code_prefix','customer_state']],on='customer_id',how='left')
    .merge(sellers[['seller_id','seller_zip_code_prefix','seller_state']],on='seller_id',how='left')
    .merge(zip_info.add_prefix('c_'),left_on='customer_zip_code_prefix',right_index=True,how='left')
    .merge(zip_info.add_prefix('s_'),left_on='seller_zip_code_prefix',right_index=True,how='left')
  )
print(items.shape,' -> ',df.shape)

(112650, 7)  ->  (112650, 24)


> **Kết luận:** ghép 7 bảng giữ đúng 112,650 dòng, không nhân và không mất dòng; số cột tăng từ 7 lên 24.

### 3.4 Xử lý giá trị thiếu

**Mục đích:** xử lý các giá trị thiếu còn lại trong bảng đã ghép, gồm hai nguồn:

- Thiếu có sẵn từ bảng products (3.1.3): danh mục, cân nặng và kích thước.
- Thiếu mới phát sinh khi ghép: tọa độ khách hàng và tọa độ người bán.

**Lưu ý:**

- Ba cột ngày thiếu của orders (3.1.3) không nằm trong df vì 3.3 chỉ lấy order_purchase_timestamp, nên không xử lý ở mục này.
- Số dòng loại ở 3.4 tính trên toàn bộ items, trước khi lọc phạm vi ở 3.5; một phần các dòng này thuộc đơn chưa giao hoặc ngoài khoảng thời gian.

#### 3.4.1 Xác định thành phần thiếu

**Câu hỏi:** sau khi ghép, cột nào của df có giá trị thiếu và thiếu bao nhiêu dòng?

**Cách làm:** đếm giá trị thiếu ở mọi cột của df, chỉ in các cột có thiếu.

In [ ]:
miss=df.isnull().sum()
print(miss[miss>0])

product_category_name            1603
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm                   18
product_category_name_english    1603
c_lat                             303
c_lng                             303
s_lat                             253
s_lng                             253
dtype: int64


> **Kết luận:** df có 10 cột thiếu giá trị, chia thành 4 nhóm theo nguyên nhân; 14 cột còn lại không thiếu.
>
> - Danh mục: product_category_name và product_category_name_english cùng thiếu 1,603 dòng; tên tiếng Anh không thiếu thêm so với tên gốc, nên cat_map_new đã vá đủ bản dịch.
> - Cân nặng và kích thước: product_weight_g, product_length_cm, product_height_cm, product_width_cm cùng thiếu 18 dòng.
> - Tọa độ khách hàng: c_lat, c_lng cùng thiếu 303 dòng.
> - Tọa độ người bán: s_lat, s_lng cùng thiếu 253 dòng.

#### 3.4.2 Xử lý thiếu tọa độ

**Cách làm:**

- Đếm số dòng có mã bưu chính không nằm trong zip_info để xác nhận nguồn gốc của giá trị thiếu.
- Tính tọa độ trung vị theo bang từ geo_in.
- Tạo cột geo_imputed đánh dấu các dòng thiếu tọa độ, rồi điền bằng trung vị của bang tương ứng.

**Lưu ý:**

- Điền thay cho loại bỏ để giữ lại dữ liệu.
- Trung vị theo bang chỉ dùng tọa độ của geo, không dùng freight_value nên không gây rò rỉ dữ liệu.
- Tọa độ điền là tâm của bang nên khoảng cách tính từ các dòng này kém chính xác hơn; cột geo_imputed đánh dấu các dòng đó.

**Bước 1:** đếm số dòng có mã bưu chính không nằm trong zip_info ở khách hàng và người bán, và tổng số dòng thiếu tọa độ.

In [ ]:
print('khach hang khong co trong zip_info:', (~df['customer_zip_code_prefix'].isin(zip_info.index)).sum())
print('nguoi ban khong co trong zip_info:', (~df['seller_zip_code_prefix'].isin(zip_info.index)).sum())
print('dong thieu toa do:', df[['c_lat', 's_lat']].isna().any(axis=1).sum())
print('ty le:', round(df[['c_lat', 's_lat']].isna().any(axis=1).mean() * 100, 3), '%')

khach hang khong co trong zip_info: 303
nguoi ban khong co trong zip_info: 253
dong thieu toa do: 555
ty le: 0.493 %


> **Kết luận:** thiếu tọa độ do mã bưu chính không có trong zip_info.
>
> - Khách hàng: 303 dòng có mã bưu chính không nằm trong zip_info, khớp 303 dòng thiếu c_lat, c_lng ở 3.4.1.
> - Người bán: 253 dòng có mã bưu chính không nằm trong zip_info, khớp 253 dòng thiếu s_lat, s_lng ở 3.4.1.
> - Số dòng thiếu tọa độ ở ít nhất một phía: 555 (0.493%).

**Bước 2:** tạo cột geo_imputed đánh dấu các dòng thiếu tọa độ, rồi điền tọa độ thiếu bằng trung vị theo bang của geo_in.

In [ ]:
df['geo_imputed']=df[['c_lat','c_lng','s_lat','s_lng']].isna().any(axis=1).astype(int)
print('geo_imputed:', df['geo_imputed'].sum())
print()
state_info=geo_in.groupby('geolocation_state')[['geolocation_lat','geolocation_lng']].median()
state_info.columns=['lat','lng']
df['c_lat']=df['c_lat'].fillna(df['customer_state'].map(state_info['lat']))
df['c_lng']=df['c_lng'].fillna(df['customer_state'].map(state_info['lng']))
df['s_lat']=df['s_lat'].fillna(df['seller_state'].map(state_info['lat']))
df['s_lng']=df['s_lng'].fillna(df['seller_state'].map(state_info['lng']))
print(df[['c_lat','c_lng','s_lat','s_lng']].isna().sum())

geo_imputed: 555

c_lat    0
c_lng    0
s_lat    0
s_lng    0
dtype: int64


> **Kết luận:** tọa độ thiếu đã được điền đủ bằng trung vị theo bang; 4 cột tọa độ không còn ô thiếu.
>
> - Khách hàng: c_lat, c_lng giảm từ 303 xuống 0 dòng thiếu.
> - Người bán: s_lat, s_lng giảm từ 253 xuống 0 dòng thiếu.
> - Cột geo_imputed: 555 dòng (0.493%) được đánh dấu 1, tức đã bị điền tọa độ ở ít nhất một trong hai phía.

#### 3.4.3 Xử lý thiếu cân nặng và kích thước

**Nguyên nhân:**

- product_weight_g, product_length_cm, product_height_cm, product_width_cm cùng thiếu 18 dòng (3.4.1).

**Cách làm:**

- Kiểm tra các dòng thiếu có đồng thời ở cả 4 cột hay không và thuộc bao nhiêu sản phẩm.
- Xử lý các dòng thiếu dựa trên kết quả kiểm tra.

**Bước 1:** Kiểm tra các dòng thiếu cân nặng hoặc kích thước có thiếu đồng thời cả 4 cột hay không, thuộc bao nhiêu sản phẩm và có thiếu danh mục hay không.

In [ ]:
cols=['product_weight_g','product_length_cm','product_height_cm','product_width_cm']
thieu=df[cols].isna()
print('Số dòng thiếu ít nhất một cột:', thieu.any(axis=1).sum())
print('Số dòng thiếu cả 4 cột:', thieu.all(axis=1).sum())
print('Số sản phẩm liên quan:', df.loc[thieu.any(axis=1),'product_id'].nunique())
print('Số dòng thiếu thêm danh mục:', df.loc[thieu.any(axis=1),'product_category_name'].isna().sum())

Số dòng thiếu ít nhất một cột: 18
Số dòng thiếu cả 4 cột: 18
Số sản phẩm liên quan: 2
Số dòng thiếu thêm danh mục: 17


> **Kết luận:** 18 dòng thiếu cân nặng hoặc kích thước đều thiếu đồng thời cả 4 cột và thuộc 2 sản phẩm.
>
> - Dòng thiếu ít nhất một cột: 18.
> - Dòng thiếu cả 4 cột: 18.
> - Số sản phẩm liên quan: 2.
> - Dòng thiếu thêm danh mục: 17.

**Bước 2:** loại bỏ các dòng thiếu cân nặng và kích thước, in số dòng trước và sau.

**Lưu ý:** loại bỏ thay cho điền vì 17 trong 18 dòng cũng thiếu danh mục (Bước 1) nên không có nhóm tham chiếu để điền; 1 dòng còn lại chiếm 0.001% nên loại cùng cho đồng nhất.

In [ ]:
keep=df[cols].notna().all(axis=1)
df=step(df,keep,'3.4.3','loai thieu can nang va kich thuoc')

3.4.3        loai thieu can nang va kich thuoc                   112,650 ->  112,632 (loai 18)


> **Kết luận:** đã loại bỏ 18 dòng thiếu cân nặng và kích thước; df còn 112,632 dòng.
>
> - Số dòng trước: 112,650.
> - Số dòng sau: 112,632.
> - Số dòng loại bỏ: 18 (khoảng 0.016%).

#### 3.4.4 Xử lý thiếu danh mục sản phẩm

**Nguyên nhân:**

- product_category_name thiếu 1,603 dòng trước 3.4.3 (3.4.1); sau khi loại 18 dòng ở 3.4.3, cần xác định số dòng còn thiếu và quy luật thiếu.

**Cách làm:**

- Đếm số dòng thiếu danh mục hiện tại và số sản phẩm liên quan.
- Xử lý các dòng thiếu dựa trên kết quả kiểm tra.

**Bước 1:** đếm số dòng thiếu product_category_name, số sản phẩm khác nhau trong các dòng đó và so sánh số dòng, trung bình, trung vị freight_value giữa nhóm thiếu danh mục với nhóm còn lại.

In [ ]:
mask=df['product_category_name'].isna()
print('Số dòng thiếu danh mục:', mask.sum())
print('Số sản phẩm liên quan:', df.loc[mask,'product_id'].nunique())
print('Số dòng, trung bình và trung vị freight_value (False: còn danh mục, True: thiếu danh mục):')
print(df.groupby(mask)['freight_value'].agg(['count','mean','median']))

Số dòng thiếu danh mục: 1586
Số sản phẩm liên quan: 609
Số dòng, trung bình và trung vị freight_value (False: còn danh mục, True: thiếu danh mục):
                        count       mean  median
product_category_name                           
False                  111046  20.025149   16.30
True                     1586  17.605025   15.23


> **Kết luận:** 1,586 dòng thiếu danh mục thuộc 609 sản phẩm, trung bình khoảng 2.6 dòng mỗi sản phẩm.
>
> - Số dòng thiếu danh mục: 1,586.
> - Số sản phẩm liên quan: 609.
> - freight_value nhóm thiếu danh mục (1,586 dòng): trung bình 17.61, trung vị 15.23.
> - freight_value nhóm còn danh mục (111,046 dòng): trung bình 20.03, trung vị 16.30.

**Bước 2:** loại bỏ các dòng thiếu danh mục, in số dòng trước và sau.

**Lưu ý:** loại bỏ thay cho gắn nhãn unknown vì danh mục là thuộc tính của sản phẩm và 609 sản phẩm này không có thông tin để suy ra danh mục (Bước 1); nhóm thiếu danh mục có freight_value thấp hơn nhóm còn lại (trung bình 17.61 so với 20.03) nhưng chỉ chiếm 1,586 dòng (1.4%). Bước này thực hiện sau 3.4.3 để 17 dòng thiếu cả danh mục lẫn kích thước đã được loại trước, số dòng ở đây là 1,586 thay vì 1,603.

In [ ]:
keep=df['product_category_name'].notna()
df=step(df,keep,'3.4.4','loai thieu danh muc')

3.4.4        loai thieu danh muc                                 112,632 ->  111,046 (loai 1,586)


> **Kết luận:** đã loại bỏ 1,586 dòng thiếu danh mục; df còn 111,046 dòng.
>
> - Số dòng trước: 112,632.
> - Số dòng sau: 111,046.
> - Số dòng loại bỏ: 1,586 (khoảng 1.4%).

#### 3.4.5 Kiểm tra lại giá trị thiếu

**Mục đích:** xác nhận df không còn cột nào thiếu giá trị sau 3.4.2, 3.4.3 và 3.4.4.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In số ô thiếu của từng cột trong df.

In [ ]:
miss=df.isnull().sum()
print(miss)

order_id                         0
order_item_id                    0
product_id                       0
seller_id                        0
shipping_limit_date              0
price                            0
freight_value                    0
customer_id                      0
order_status                     0
order_purchase_timestamp         0
product_category_name            0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
customer_zip_code_prefix         0
customer_state                   0
seller_zip_code_prefix           0
seller_state                     0
c_lat                            0
c_lng                            0
s_lat                            0
s_lng                            0
geo_imputed                      0
dtype: int64


> **Kết luận:** df không còn giá trị thiếu ở cả 25 cột.

### 3.5 Xác định phạm vi dữ liệu sử dụng

**Mục đích:** giới hạn dữ liệu ở các đơn hàng đã giao và khoảng thời gian phân tích.

#### 3.5.1 Chỉ giữ đơn hàng đã giao thành công

**Câu hỏi:** các dòng thuộc đơn ở trạng thái nào được dùng để dự đoán phí vận chuyển?

**Cách làm:**

- Đếm số dòng theo order_status.
- Chỉ giữ trạng thái delivered.

**Bước 1:** đếm số dòng theo order_status.

In [ ]:
print(df['order_status'].value_counts())

order_status
delivered      108659
shipped          1158
canceled          528
invoiced          347
processing        344
unavailable         7
approved            3
Name: count, dtype: int64


> **Kết luận:** 108,659 trong 111,046 dòng thuộc đơn delivered; 2,387 dòng thuộc 6 trạng thái còn lại.
>
> - shipped: 1,158 dòng.
> - canceled: 528 dòng.
> - invoiced: 347 dòng.
> - processing: 344 dòng.
> - unavailable: 7 dòng.
> - approved: 3 dòng.

**Bước 2:** chỉ giữ các dòng có order_status = delivered, in số dòng trước và sau.

**Lưu ý:** các đơn chưa giao hoặc bị hủy chưa hoàn tất giao dịch nên không dùng để học phí vận chuyển; bước này thực hiện trước 3.5.2 để số đếm theo tháng ở 3.5.2 chỉ tính các đơn đã giao.

In [ ]:
keep_delivered = df['order_status'] == 'delivered'

df = step(
    df,
    keep_delivered,
    '3.5.1',
    'chi giu order_status = delivered'
)

print(df['order_status'].value_counts())

3.5.1        chi giu order_status = delivered                    111,046 ->  108,659 (loai 2,387)
order_status
delivered    108659
Name: count, dtype: int64


> **Kết luận:** loại 2,387 dòng thuộc đơn chưa hoàn thành hoặc bị hủy; df còn 108,659 dòng, đều thuộc đơn delivered.

#### 3.5.2 Giới hạn khoảng thời gian phân tích

**Câu hỏi:** giai đoạn nào có quá ít đơn hàng so với phần còn lại?

**Cách làm:**

- Chuyển order_purchase_timestamp sang datetime.
- Đếm số dòng theo tháng.
- Giữ các giao dịch có order_purchase_timestamp từ 01/01/2017 đến trước 01/09/2018.

**Lưu ý:** bước này thực hiện sau 3.5.1 để số đếm theo tháng chỉ tính các đơn đã giao.

**Bước 1:** chuyển cột order_purchase_timestamp sang kiểu datetime để lọc theo khoảng thời gian.

In [ ]:
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

**Bước 2:** đếm số dòng theo tháng mua hàng để xác định các tháng có ít quan sát.

In [ ]:
print(df['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index())

order_purchase_timestamp
2016-09       3
2016-10     311
2016-12       1
2017-01     900
2017-02    1802
2017-03    2837
2017-04    2500
2017-05    3913
2017-06    3431
2017-07    4358
2017-08    4722
2017-09    4664
2017-10    5146
2017-11    8357
2017-12    6050
2018-01    7844
2018-02    7401
2018-03    7884
2018-04    7753
2018-05    7768
2018-06    6987
2018-07    6916
2018-08    7111
Freq: M, Name: count, dtype: int64


**Bước 3:** giữ các giao dịch từ 01/01/2017 đến hết 31/08/2018, in khoảng thời gian trước và sau khi lọc.

In [ ]:
print('thoi gian truoc:')
print(df['order_purchase_timestamp'].min())
print(df['order_purchase_timestamp'].max())

keep_time = (
    (df['order_purchase_timestamp'] >= '2017-01-01') &
    (df['order_purchase_timestamp'] < '2018-09-01')
)

df = step(
    df,
    keep_time,
    '3.5.2',
    'thoi gian 01/2017 den 08/2018'
)

print()
print('thoi gian sau:')
print(df['order_purchase_timestamp'].min())
print(df['order_purchase_timestamp'].max())

thoi gian truoc:
2016-09-15 12:16:38
2018-08-29 15:00:37
3.5.2        thoi gian 01/2017 den 08/2018                       108,659 ->  108,344 (loai 315)

thoi gian sau:
2017-01-05 11:56:06
2018-08-29 15:00:37


> **Kết luận:** loại 315 dòng, tất cả thuộc năm 2016; df còn 108,344 dòng.
>
> - Số dòng theo tháng của năm 2016: 2016-09 có 3 dòng, 2016-10 có 311 dòng, 2016-12 có 1 dòng.
> - Thời gian nhỏ nhất sau lọc: 2017-01-05.
> - Thời gian lớn nhất: 2018-08-29; không có dòng nào từ 01/09/2018 trở đi nên cận trên không loại dòng nào.

### 3.6 Xử lý giá trị sai miền

**Mục đích:** kiểm tra và loại các giá trị không hợp lệ còn lại sau khi ghép và lọc phạm vi.

#### 3.6.1 Kiểm tra giá trị sai miền sau khi ghép và lọc phạm vi

**Mục đích:** đếm các giá trị không hợp lệ về mặt vật lý hoặc nghiệp vụ trên bảng đã ghép; việc loại thực hiện ở 3.6.2.

**Các điều kiện kiểm tra:**

- price phải lớn hơn 0.
- freight_value không được âm.
- product_weight_g phải lớn hơn 0.
- Chiều dài, chiều cao và chiều rộng phải lớn hơn 0.

**Lưu ý:** freight_value = 0 nằm trong miền hợp lệ nên không tính là sai miền; việc có loại các dòng này hay không được xem xét ở 3.8. Kiểm tra lại ở mức order-item vì 3.1.5 đếm weight ở mức bảng products.

In [ ]:
print('price <= 0       :', (df['price'] <= 0).sum())
print('freight < 0      :', (df['freight_value'] < 0).sum())
print('freight = 0      :', (df['freight_value'] == 0).sum())

print('weight <= 0      :', (df['product_weight_g'] <= 0).sum())
print('length <= 0      :', (df['product_length_cm'] <= 0).sum())
print('height <= 0      :', (df['product_height_cm'] <= 0).sum())
print('width <= 0       :', (df['product_width_cm'] <= 0).sum())
print('so san pham weight <= 0:', df.loc[df['product_weight_g'] <= 0, 'product_id'].nunique())

price <= 0       : 0
freight < 0      : 0
freight = 0      : 379
weight <= 0      : 8
length <= 0      : 0
height <= 0      : 0
width <= 0       : 0
so san pham weight <= 0: 4


> **Kết luận:** chỉ product_weight_g có giá trị sai miền; price, freight_value và ba kích thước không có.
>
> - product_weight_g <= 0: 8 dòng, thuộc 4 sản phẩm (3.1.5 đếm 4 dòng ở bảng products, đơn vị là sản phẩm).
> - price <= 0, freight_value < 0, chiều dài, chiều cao, chiều rộng <= 0: 0 dòng.
> - freight_value = 0: 379 dòng, nằm trong miền hợp lệ và được xem xét ở 3.8.

#### 3.6.2 Loại giá trị vật lý không hợp lệ

**Nguyên nhân:**

- product_weight_g = 0 ở 8 dòng (3.6.1); sản phẩm vật lý dùng trong vận chuyển không thể có cân nặng bằng 0 nên đây là giá trị sai miền.

**Cách làm:**

- Tạo một điều kiện hợp lệ chung cho price, freight_value, cân nặng và ba kích thước.
- Lọc bằng hàm step().

In [ ]:
valid_value = (
    (df['price'] > 0) &
    (df['freight_value'] >= 0) &
    (df['product_weight_g'] > 0) &
    (df['product_length_cm'] > 0) &
    (df['product_height_cm'] > 0) &
    (df['product_width_cm'] > 0)
)

print('Số dòng hợp lệ:', valid_value.sum())

df = step(
    df,
    valid_value,
    '3.6.2',
    'loai gia tri sai mien'
)

print('so dong con lai:', len(df))

Số dòng hợp lệ: 108336
3.6.2        loai gia tri sai mien                               108,344 ->  108,336 (loai 8)
so dong con lai: 108336


> **Kết luận:** loại 8 dòng có product_weight_g = 0; df còn 108,336 dòng.
>
> - Không có dòng nào bị loại vì price, freight_value hoặc ba kích thước.

### 3.7 Kiểm tra tính nhất quán sau làm sạch

**Mục đích:** kiểm tra các bước xử lý không làm thay đổi grain của dữ liệu và không tạo lỗi mới.

**Lưu ý:** grain của bảng cuối vẫn là một sản phẩm trong một đơn hàng, xác định bởi cặp (order_id, order_item_id).

**Cách làm:**

- Kiểm tra thứ tự các mốc ngày trên bảng orders (chỉ ghi nhận, không loại dòng).
- Kiểm tra cặp khóa có bị trùng không.
- Kiểm tra còn giá trị thiếu không.
- Kiểm tra toàn bộ đơn đều ở trạng thái delivered.
- Kiểm tra khoảng thời gian.
- Kiểm tra lại miền giá trị của biến mục tiêu và các biến vật lý.

**Bước 1:** kiểm tra thứ tự ngày trên các đơn delivered của bảng orders; chỉ ghi nhận số dòng vi phạm, không loại dòng vì chỉ order_purchase_timestamp được dùng làm đặc trưng.

In [ ]:
od = orders[orders['order_status'] == 'delivered'].copy()
od['order_purchase_timestamp'] = pd.to_datetime(od['order_purchase_timestamp'])
od['order_approved_at'] = pd.to_datetime(od['order_approved_at'])
od['order_delivered_carrier_date'] = pd.to_datetime(od['order_delivered_carrier_date'])
od['order_delivered_customer_date'] = pd.to_datetime(od['order_delivered_customer_date'])

print('duyet < mua:', (od['order_approved_at'] < od['order_purchase_timestamp']).sum())
print('giao khach < mua:', (od['order_delivered_customer_date'] < od['order_purchase_timestamp']).sum())
print('giao khach < giao van chuyen:', (od['order_delivered_customer_date'] < od['order_delivered_carrier_date']).sum())

duyet < mua: 0
giao khach < mua: 0
giao khach < giao van chuyen: 23


> **Kết luận:** các đơn delivered không có đơn nào được duyệt hoặc giao đến khách trước ngày mua; 23 đơn có ngày giao đến khách sớm hơn ngày giao cho đơn vị vận chuyển.
>
> - Ngày duyệt nhỏ hơn ngày mua: 0 đơn.
> - Ngày giao đến khách nhỏ hơn ngày mua: 0 đơn.
> - Ngày giao đến khách nhỏ hơn ngày giao cho đơn vị vận chuyển: 23 đơn.
> - Chỉ ghi nhận, không loại dòng vì phần sau chỉ dùng order_purchase_timestamp làm đặc trưng.

**Bước 2:** kiểm tra lại grain, giá trị thiếu, trạng thái, khoảng thời gian và miền giá trị trên df.

In [ ]:
print(
    'trung grain:',
    df[['order_id', 'order_item_id']].duplicated().sum()
)

print(
    'tong missing:',
    df.isna().sum().sum()
)

print(
    'khac delivered:',
    (df['order_status'] != 'delivered').sum()
)

print(
    'ngoai thoi gian:',
    (
        (df['order_purchase_timestamp'] < '2017-01-01') |
        (df['order_purchase_timestamp'] >= '2018-09-01')
    ).sum()
)

print(
    'price <= 0:',
    (df['price'] <= 0).sum()
)

print(
    'freight < 0:',
    (df['freight_value'] < 0).sum()
)

print(
    'weight <= 0:',
    (df['product_weight_g'] <= 0).sum()
)

print(
    'kich thuoc <= 0:',
    (
        (df['product_length_cm'] <= 0) |
        (df['product_height_cm'] <= 0) |
        (df['product_width_cm'] <= 0)
    ).sum()
)

trung grain: 0
tong missing: 0
khac delivered: 0
ngoai thoi gian: 0
price <= 0: 0
freight < 0: 0
weight <= 0: 0
kich thuoc <= 0: 0


> **Kết luận:** dữ liệu sau làm sạch thỏa cả 8 điều kiện kiểm tra.
>
> - Trùng grain (order_id, order_item_id): 0 dòng.
> - Giá trị thiếu: 0 ô.
> - Trạng thái khác delivered: 0 dòng.
> - Ngoài khoảng 01/2017 đến 08/2018: 0 dòng.
> - price <= 0, freight_value < 0, product_weight_g <= 0, kích thước <= 0: mỗi điều kiện 0 dòng.

### 3.8 Business rule đối với freight_value = 0

**Câu hỏi:** các dòng freight_value = 0 có cần loại khỏi dữ liệu làm sạch hay không?

**Cách làm:**

- Đếm số dòng và tỷ lệ freight_value = 0.
- Đếm số người bán, phân bố theo tháng và so sánh trung vị cân nặng, giá của các dòng này.
- So sánh tỷ lệ freight_value = 0 trong nhóm người bán có dòng này với phần còn lại của họ.

**Lưu ý:**

- freight_value = 0 nằm trong miền hợp lệ (3.6.1), nên không xếp vào lỗi sai miền.
- Các dòng này thuộc đơn đã giao thành công (3.5.1) và đã đủ các trường khác sau 3.4.
- Làm sạch chỉ loại giá trị sai hoặc không thể suy luận; loại dòng theo giá trị của biến mục tiêu làm thay đổi phân phối freight_value và mô hình có thể chỉ học cho phần phí lớn hơn 0.
- Giữ dòng là quyết định có thể đảo ngược: ở bước sau chỉ cần một điều kiện freight_value > 0 để loại. Ngược lại, loại ngay ở bước làm sạch thì dữ liệu đã xuất không còn các dòng này.
- Quyết định loại, nếu cần, được xem xét sau khi khám phá dữ liệu.

**Bước 1:** đếm số dòng và tỷ lệ freight_value = 0.

In [ ]:
zero_freight = (df['freight_value'] == 0).sum()

print('freight_value = 0:', zero_freight)
print(
    'ty le:',
    round(zero_freight / len(df) * 100, 3),
    '%'
)
print('freight_value = 0 o items goc:', (items['freight_value'] == 0).sum())

freight_value = 0: 379
ty le: 0.35 %
freight_value = 0 o items goc: 383


**Bước 2:** kiểm tra các dòng freight_value = 0 theo người bán, theo tháng mua hàng và theo cân nặng, giá.

In [ ]:
zero = df[df['freight_value'] == 0]

print('so nguoi ban:', zero['seller_id'].nunique())
print()
print(zero['order_purchase_timestamp'].dt.to_period('M').value_counts().sort_index())
print()
print(df.groupby(df['freight_value'] == 0)[['product_weight_g', 'price']].median())

so nguoi ban: 8

order_purchase_timestamp
2017-08     10
2017-09      1
2018-04    151
2018-05    215
2018-08      2
Freq: M, Name: count, dtype: int64

               product_weight_g  price
freight_value                         
False                     700.0   74.9
True                      584.0   99.9


**Bước 3:** với tám người bán có dòng freight_value = 0, tính tỷ lệ dòng freight_value = 0 theo tháng trong toàn bộ dòng của họ.

**Lưu ý:** bước này đặt sau Bước 2 vì cần danh sách người bán và tháng tập trung đã xác định ở đó. Tỷ lệ freight_value = 0 chỉ tập trung ở hai tháng và gần 0 ở các tháng còn lại cho thấy các dòng này gắn với thời điểm, nhưng chưa đủ để kết luận nguyên nhân.

In [ ]:
nguoi_ban_zero = zero['seller_id'].unique()
sub = df[df['seller_id'].isin(nguoi_ban_zero)]

print('so dong cua cac nguoi ban nay:', len(sub))
print('so dong freight_value = 0   :', (sub['freight_value'] == 0).sum())
print()
print((sub['freight_value'] == 0).groupby(sub['order_purchase_timestamp'].dt.to_period('M')).mean().round(3))

so dong cua cac nguoi ban nay: 7563
so dong freight_value = 0   : 379

order_purchase_timestamp
2017-01    0.000
2017-02    0.000
2017-03    0.000
2017-04    0.000
2017-05    0.000
2017-06    0.000
2017-07    0.000
2017-08    0.022
2017-09    0.002
2017-10    0.000
2017-11    0.000
2017-12    0.000
2018-01    0.000
2018-02    0.000
2018-03    0.000
2018-04    0.217
2018-05    0.285
2018-06    0.000
2018-07    0.000
2018-08    0.007
Freq: M, Name: freight_value, dtype: float64


> **Kết luận:** 379 dòng freight_value = 0 được giữ lại trong dữ liệu làm sạch; các dòng này hợp lệ về miền giá trị, thuộc đơn đã giao thành công và chưa có đủ căn cứ để coi là lỗi.
>
> - Số dòng: 379 (0.35%).
> - Số người bán: 8.
> - Tháng tập trung: 2018-04 (151 dòng) và 2018-05 (215 dòng), cộng lại 366 trong 379 dòng.
> - Trung vị cân nặng 584 và giá 99.9 ở nhóm freight_value = 0, so với 700 và 74.9 ở nhóm còn lại.
> - Trong toàn bộ 7,563 dòng của 8 người bán này, 379 dòng có freight_value = 0 (5.01%).
> - Tỷ lệ freight_value = 0 của 8 người bán là 21.7% ở 2018-04 và 28.5% ở 2018-05, không quá 2.2% ở các tháng còn lại.
> - Số dòng freight_value = 0 ở bảng items gốc là 383 (3.1.5), còn 379 sau các bước lọc ở 3.4 đến 3.6.2.
> - Việc loại các dòng này, nếu cần, được quyết định sau khi khám phá dữ liệu.
> - Số dòng của df sau 3.8: 108,336.

### 3.9 Tóm tắt dữ liệu cuối

**Mục đích:** ghi nhận kích thước và thống kê biến mục tiêu của dataset trước khi xuất file.

**Cách làm:** in kích thước cuối, số dòng freight_value = 0 và thống kê mô tả của freight_value.

In [ ]:
print('kich thuoc cuoi:', df.shape)
print('so dong freight_value = 0:', (df['freight_value'] == 0).sum())
print()
print(df['freight_value'].describe())

kich thuoc cuoi: (108336, 25)
so dong freight_value = 0: 379

count    108336.000000
mean         19.983661
std          15.738508
min           0.000000
25%          13.080000
50%          16.290000
75%          21.160000
max         409.680000
Name: freight_value, dtype: float64


> **Kết luận:** dataset cuối có 108,336 dòng và 25 cột; freight_value có trung bình 19.98, trung vị 16.29, nhỏ nhất 0.00, lớn nhất 409.68.
>
> - Số dòng freight_value = 0: 379, được giữ lại ở 3.8.

### 3.10 Audit trail

**Mục đích:** tổng hợp các bước loại dòng khỏi df và kiểm tra tổng số dòng loại khớp với số dòng cuối.

**Cách làm:** hàm step() đã lưu số dòng trước và sau mỗi lần lọc vào audit_rows; chuyển danh sách này thành DataFrame để kiểm tra và dùng trong báo cáo.

In [ ]:
audit = pd.DataFrame(audit_rows)

print(audit)
print('Tổng số dòng loại:', audit['loai'].sum())
print('Khớp số dòng cuối:', audit['truoc'].iloc[0] - audit['loai'].sum() == audit['sau'].iloc[-1])
print('Tỉ lệ dòng loại:', round(audit['loai'].sum() / audit['truoc'].iloc[0] * 100, 2), '%')

     muc                            quy_tac   truoc     sau  loai
0  3.4.3  loai thieu can nang va kich thuoc  112650  112632    18
1  3.4.4                loai thieu danh muc  112632  111046  1586
2  3.5.1   chi giu order_status = delivered  111046  108659  2387
3  3.5.2      thoi gian 01/2017 den 08/2018  108659  108344   315
4  3.6.2              loai gia tri sai mien  108344  108336     8
Tổng số dòng loại: 4314
Khớp số dòng cuối: True
Tỉ lệ dòng loại: 3.83 %


> **Kết luận:** 5 bước loại tổng cộng 4,314 dòng (3.83%), từ 112,650 xuống 108,336; tổng khớp với số dòng cuối.
>
> - 3.4.3: 18 dòng.
> - 3.4.4: 1,586 dòng.
> - 3.5.1: 2,387 dòng.
> - 3.5.2: 315 dòng.
> - 3.6.2: 8 dòng.

### 3.11 Xuất dữ liệu sạch

**Mục đích:** lưu dataset đã làm sạch để dùng cho các bước SQL, EDA và xây dựng mô hình mà không phải chạy lại toàn bộ quy trình.

**Cách làm:**

- Ghi df ra olist_merged_clean.csv (dataset chính sau làm sạch).
- Ghi audit ra audit_trail.csv (lịch sử các bước lọc dữ liệu).
- Đọc lại tệp đã lưu để so kích thước, số ô thiếu và số dòng trùng grain với df.

**Lưu ý:** dùng index=False để không tạo thêm cột index khi ghi CSV.

In [ ]:
import os

output_dir = '/content/drive/MyDrive/Colab Notebooks/output/'

os.makedirs(output_dir, exist_ok=True)

output_data = output_dir + 'olist_merged_clean.csv'
output_audit = output_dir + 'audit_trail.csv'

audit = pd.DataFrame(audit_rows)

df.to_csv(
    output_data,
    index=False,
    encoding='utf-8-sig'
)

audit.to_csv(
    output_audit,
    index=False,
    encoding='utf-8-sig'
)

print('Da xuat du lieu:')
print(output_data)
print(output_audit)

check = pd.read_csv(output_data, parse_dates=['order_purchase_timestamp'])
print('df hien tai:', df.shape)
print('file da luu:', check.shape)
print('tong missing:', check.isna().sum().sum())
print('trung grain:', check[['order_id', 'order_item_id']].duplicated().sum())

Da xuat du lieu:
/content/drive/MyDrive/Colab Notebooks/output/olist_merged_clean.csv
/content/drive/MyDrive/Colab Notebooks/output/audit_trail.csv
df hien tai: (108336, 25)
file da luu: (108336, 25)
tong missing: 0
trung grain: 0


> **Kết luận:** file olist_merged_clean.csv có 108,336 dòng và 25 cột, khớp với df; không còn ô thiếu và không trùng grain.

---

## 4. Chuyển đổi dữ liệu

**Mục đích:** sau khi làm sạch, chuyển đổi kiểu dữ liệu và cấu trúc bảng để phục vụ phân tích và mô hình hoá.

In kích thước và kiểu dữ liệu của từng cột trong df để xác định cột nào cần chuyển kiểu ở 4.1.

In [ ]:
print(df.shape)
print()
print(df.dtypes)

(108336, 25)

order_id                                 object
order_item_id                             int64
product_id                               object
seller_id                                object
shipping_limit_date                      object
price                                   float64
freight_value                           float64
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
product_category_name                    object
product_weight_g                        float64
product_length_cm                       float64
product_height_cm                       float64
product_width_cm                        float64
product_category_name_english            object
customer_zip_code_prefix                  int64
customer_state                           object
seller_zip_code_prefix                    int64
seller_state                             object
c_lat                     

### 4.1 Chuyển kiểu dữ liệu

Chuyển order_item_id sang số nguyên và product_weight_g sang số thực, rồi in kiểu của hai cột này cùng order_purchase_timestamp.

In [ ]:
df['order_item_id'] = df['order_item_id'].astype(int)
df['product_weight_g'] = df['product_weight_g'].astype(float)
print(df.dtypes[['order_item_id', 'product_weight_g', 'order_purchase_timestamp']])

order_item_id                        int64
product_weight_g                   float64
order_purchase_timestamp    datetime64[ns]
dtype: object


> **Kết luận:** order_item_id, product_weight_g và order_purchase_timestamp đã ở kiểu phù hợp.
>
> - order_item_id: int64.
> - product_weight_g: float64.
> - order_purchase_timestamp: datetime64[ns], đã chuyển ở 3.5.2.

### 4.2 Chuyển đổi ngày

**Mục đích:** từ order_purchase_timestamp tạo hai đặc trưng thời gian để chia dữ liệu theo thời gian ở bước mô hình hoá.

**Cách làm:**

- t_days: số ngày kể từ mốc 01/01/2017.
- month: nhãn tháng dạng YYYY-MM.

**Lưu ý:** mốc 01/01/2017 là mốc cố định, trùng với đầu khoảng thời gian giữ lại ở 3.5.2 và không tính từ dữ liệu, nên t_days không gây rò rỉ dữ liệu.



**Bước 1:** tạo t_days bằng hiệu giữa order_purchase_timestamp và mốc 01/01/2017, lấy phần số ngày.

In [ ]:
T0=pd.to_datetime('2017-01-01')
df['t_days'] = (df['order_purchase_timestamp'] - T0).dt.days
print(df['t_days'].min())
print(df['t_days'].max())

4
605


> **Kết luận:** t_days nhận giá trị từ 4 đến 605 ngày.
>
> - Giá trị nhỏ nhất: 4, ứng với đơn mua ngày 2017-01-05.
> - Giá trị lớn nhất: 605, ứng với đơn mua ngày 2018-08-29.

**Bước 2:** tạo month bằng cách lấy tháng của order_purchase_timestamp rồi chuyển sang chuỗi.

In [ ]:
df['month']=df['order_purchase_timestamp'].dt.to_period('M').astype(str)
print(df['month'].min())
print(df['month'].max())
print(df['month'].nunique())

2017-01
2018-08
20


> **Kết luận:** month nhận giá trị từ 2017-01 đến 2018-08, gồm 20 tháng khác nhau.
>
> - Tháng nhỏ nhất: 2017-01.
> - Tháng lớn nhất: 2018-08.
> - Số tháng khác nhau: 20.

### 4.3 Ghép các bảng

**Mục đích:** bổ sung cột order_n_items, là số sản phẩm của mỗi đơn hàng, vào bảng đã ghép ở 3.3.

**Lưu ý:** order_n_items được đếm trên bảng items gốc để phản ánh kích thước thật của đơn; nếu đếm trên df thì các dòng đã bị loại ở phần 3 làm số đếm thấp hơn thực tế.

In [ ]:
print('truoc:', df.shape)
order_n_items = items.groupby('order_id').size()
df['order_n_items'] = df['order_id'].map(order_n_items)
print('sau  :', df.shape)

truoc: (108336, 27)
sau  : (108336, 28)


> **Kết luận:** đã thêm cột order_n_items vào df; số cột tăng từ 27 lên 28 và số dòng giữ nguyên.
>
> - Trước khi thêm: 108,336 dòng, 27 cột (25 cột ban đầu cùng t_days và month).
> - Sau khi thêm: 108,336 dòng, 28 cột.

### 4.4 Xử lý quan hệ 1-N

**Câu hỏi:** freight_value là phí riêng của từng sản phẩm hay phí của cả đơn được chia cho các sản phẩm?

**Cách làm:**

- Chia df thành ba nhóm: đơn một sản phẩm, đơn nhiều sản phẩm cùng một người bán, đơn nhiều sản phẩm có từ hai người bán trở lên.
- Tính tương quan giữa freight_value và product_weight_g ở mỗi nhóm.

**Lưu ý:**

- Nếu phí là tổng của cả đơn chia đều thì ở đơn nhiều sản phẩm, phí không còn phụ thuộc cân nặng của từng sản phẩm, nên tương quan ở nhóm này sẽ thấp hơn rõ so với nhóm một sản phẩm.
- Ở nhóm cùng một người bán, các sản phẩm có cùng khoảng cách giao nên chỉ cân nặng khác nhau; nhóm này dùng để kết luận.
- Nhóm có từ hai người bán trở lên có khoảng cách giao khác nhau giữa các dòng, chỉ ghi nhận để tham khảo.

In [ ]:
so_nguoi_ban = df.groupby('order_id')['seller_id'].transform('nunique')
single = df[df['order_n_items'] == 1]
multi_same = df[(df['order_n_items'] > 1) & (so_nguoi_ban == 1)]
multi_diff = df[(df['order_n_items'] > 1) & (so_nguoi_ban > 1)]

corr_single = single['freight_value'].corr(single['product_weight_g'])
corr_same = multi_same['freight_value'].corr(multi_same['product_weight_g'])
corr_diff = multi_diff['freight_value'].corr(multi_diff['product_weight_g'])

print('don 1 san pham                      :', len(single), 'dong, corr =', f'{corr_single:.3f}')
print('don nhieu san pham, 1 nguoi ban     :', len(multi_same), 'dong, corr =', f'{corr_same:.3f}')
print('don nhieu san pham, nhieu nguoi ban :', len(multi_diff), 'dong, corr =', f'{corr_diff:.3f}')
print('chenh lech (nhom 1 va nhom 2)       :', f'{abs(corr_single - corr_same):.3f}')

don 1 san pham                      : 85370 dong, corr = 0.618
don nhieu san pham, 1 nguoi ban     : 19954 dong, corr = 0.613
don nhieu san pham, nhieu nguoi ban : 3012 dong, corr = 0.413
chenh lech (nhom 1 va nhom 2)       : 0.006


> **Kết luận:** tương quan giữa freight_value và product_weight_g ở đơn nhiều sản phẩm của cùng một người bán gần bằng ở đơn một sản phẩm, nên freight_value được xem là phí theo từng sản phẩm và grain cấp sản phẩm được giữ.
>
> - Đơn một sản phẩm: 85,370 dòng, tương quan 0.618.
> - Đơn nhiều sản phẩm, một người bán: 19,954 dòng, tương quan 0.613.
> - Đơn nhiều sản phẩm, nhiều người bán: 3,012 dòng, tương quan 0.413. Nhóm này chỉ ghi nhận, không dùng để kết luận vì khoảng cách giao giữa các dòng khác nhau.
> - Chênh lệch giữa hai nhóm dùng để kết luận: 0.006, tính từ giá trị chưa làm tròn.
> - Tổng ba nhóm: 108,336 dòng, khớp với số dòng của df.

### 4.5. Chốt grain của bảng

Chốt grain là một sản phẩm trong một đơn hàng, xác định bởi cặp (order_id, order_item_id); in số dòng, số đơn và số dòng trùng theo grain.

In [ ]:
GRAIN = ['order_id', 'order_item_id']

print('so dong:', len(df))
print('so don :', df['order_id'].nunique())
print('trung theo grain:', df.duplicated(subset=GRAIN).sum())

so dong: 108336
so don : 94874
trung theo grain: 0


> **Kết luận:** cặp (order_id, order_item_id) xác định duy nhất từng dòng của bảng nên grain cấp sản phẩm trong đơn hàng được giữ nguyên.
>
> - Số dòng: 108,336.
> - Số đơn hàng: 94,874.
> - Số dòng trùng theo grain: 0.

### 4.6. Tạo đặc trưng

**Mục đích:** tạo ba đặc trưng số cho mô hình từ các cột đã làm sạch: khoảng cách giao, thể tích sản phẩm và cờ cùng bang.

**Cách làm:**
- distance_km: khoảng cách đường chim bay giữa người bán và khách hàng, tính bằng công thức Haversine từ s_lat, s_lng, c_lat, c_lng.
- volume_cm3: tích của chiều dài, chiều cao và chiều rộng sản phẩm.
- same_state: bằng 1 nếu seller_state trùng customer_state, ngược lại bằng 0.

**Lưu ý:**
- Tọa độ đã được điền đủ ở 3.4.2 nên distance_km không có giá trị thiếu.
- Khoảng cách chim bay không phải quãng đường vận chuyển thực tế, đây là hạn chế của dữ liệu.
- Giữ cả ba cạnh thô (product_length_cm, product_height_cm, product_width_cm) bên cạnh volume_cm3.

In [ ]:
def haversine_km(lat1, lng1, lat2, lng2):
    lat1, lng1, lat2, lng2 = map(np.radians, [lat1, lng1, lat2, lng2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lng2 - lng1) / 2) ** 2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

df['distance_km'] = haversine_km(df['s_lat'], df['s_lng'], df['c_lat'], df['c_lng'])
df['volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']
df['same_state'] = (df['seller_state'] == df['customer_state']).astype(int)

print(df[['distance_km', 'volume_cm3', 'same_state']].describe())
print()
print(df.shape)

         distance_km     volume_cm3     same_state
count  108336.000000  108336.000000  108336.000000
mean      597.024956   15215.569497       0.361930
std       586.972152   23268.223084       0.480561
min         0.000000     168.000000       0.000000
25%       187.241215    2856.000000       0.000000
50%       433.968061    6525.000000       0.000000
75%       793.738686   18375.000000       1.000000
max      3398.485605  296208.000000       1.000000

(108336, 31)


> **Kết luận:** ba đặc trưng được tạo đủ cho 108,336 dòng, không có giá trị thiếu, và bảng tăng lên 31 cột.
>
> - distance_km: trung bình 597.02 km, trung vị 433.97 km, nhỏ nhất 0.00 km, lớn nhất 3,398.49 km.
> - volume_cm3: trung bình 15,215.57, trung vị 6,525, nhỏ nhất 168, lớn nhất 296,208.
> - same_state: trung bình 0.362, tức khoảng 36.2% dòng có người bán và khách hàng cùng bang.
> - distance_km và volume_cm3 có trung bình lớn hơn trung vị.

### 4.7. Đổi tên cột và chốt schema cuối

Đổi tên các cột sang tên ngắn dùng cho mô hình và bỏ các cột không còn sử dụng: order_status, shipping_limit_date, product_category_name.

In [ ]:
df = df.rename(columns={
    'product_weight_g': 'weight_g',
    'product_length_cm': 'length_cm',
    'product_height_cm': 'height_cm',
    'product_width_cm': 'width_cm',
    'product_category_name_english': 'category',
    's_lat': 'seller_lat',
    's_lng': 'seller_lng',
    'c_lat': 'customer_lat',
    'c_lng': 'customer_lng',
    'seller_zip_code_prefix': 'seller_zip',
    'customer_zip_code_prefix': 'customer_zip'
})
df = df.drop(columns=['order_status', 'shipping_limit_date', 'product_category_name'], errors='ignore')

print(df.shape)
print(list(df.columns))

(108336, 28)
['order_id', 'order_item_id', 'product_id', 'seller_id', 'price', 'freight_value', 'customer_id', 'order_purchase_timestamp', 'weight_g', 'length_cm', 'height_cm', 'width_cm', 'category', 'customer_zip', 'customer_state', 'seller_zip', 'seller_state', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'geo_imputed', 't_days', 'month', 'order_n_items', 'distance_km', 'volume_cm3', 'same_state']


> **Kết luận:** schema cuối gồm 108,336 dòng và 28 cột, số dòng không đổi so với 4.5.
>
> - Đổi tên 11 cột sang tên ngắn dùng cho mô hình.
> - Bỏ 3 cột: order_status, shipping_limit_date, product_category_name.
> - Số cột giảm từ 31 xuống 28.

## 5. Kiểm tra tính hợp lệ

**Mục đích:** xác nhận bảng cuối đáp ứng các tiêu chuẩn kỹ thuật và nghiệp vụ trước khi chuyển sang phân tích và mô hình hóa.

### 5.1. Trùng lặp theo grain

Đếm số dòng trùng theo cặp (order_id, order_item_id) trên bảng cuối.

In [ ]:
print('trung theo grain:', df.duplicated(subset=GRAIN).sum())

trung theo grain: 0


> **Kết luận:** không có dòng nào trùng theo cặp (order_id, order_item_id), số dòng trùng bằng 0.

### 5.2. Giá trị thiếu

Đếm tổng số giá trị thiếu của bảng và liệt kê các cột còn thiếu.

In [ ]:
print('tong gia tri thieu:', df.isna().sum().sum())
print('so cot con thieu  :', df.isna().any().sum())

tong gia tri thieu: 0
so cot con thieu  : 0


> **Kết luận:** bảng cuối không còn giá trị thiếu, cả tổng số giá trị thiếu và số cột còn thiếu đều bằng 0.

### 5.3. Kiểm tra trường giá

Kiểm tra khoảng giá trị của price và freight_value, đếm số dòng price không dương, freight_value âm và freight_value bằng 0.

In [ ]:
print('price min / max       :', df['price'].min(), '/', df['price'].max())
print('so dong price <= 0    :', (df['price'] <= 0).sum())
print('freight min / max     :', df['freight_value'].min(), '/', df['freight_value'].max())
print('so dong freight < 0   :', (df['freight_value'] < 0).sum())
print('so dong freight = 0   :', (df['freight_value'] == 0).sum())

price min / max       : 0.85 / 6735.0
so dong price <= 0    : 0
freight min / max     : 0.0 / 409.68
so dong freight < 0   : 0
so dong freight = 0   : 379


> **Kết luận:** price và freight_value đều nằm trong miền hợp lệ, không có dòng nào vi phạm.
>
> - price: nhỏ nhất 0.85, lớn nhất 6,735.00; số dòng price ≤ 0 là 0.
> - freight_value: nhỏ nhất 0.00, lớn nhất 409.68; số dòng âm là 0.
> - Số dòng freight_value = 0 là 379, hợp lệ và được giữ lại ở 3.8.

### 5.4. Danh mục sản phẩm

Đếm số nhóm danh mục và số dòng thiếu category.

In [ ]:
print('so nhom category      :', df['category'].nunique())
print('so dong thieu category:', df['category'].isna().sum())

so nhom category      : 73
so dong thieu category: 0


> **Kết luận:** bảng cuối có 73 nhóm danh mục và không có dòng nào thiếu category.

### 5.5. Logic thời gian

Đếm số đơn có ngày mua nằm ngoài khoảng từ 01/2017 đến 08/2018.

In [ ]:
print('mua ngoai 01/2017 - 08/2018:', ((df['order_purchase_timestamp'] < '2017-01-01') | (df['order_purchase_timestamp'] > '2018-08-31 23:59:59')).sum())

mua ngoai 01/2017 - 08/2018: 0


> **Kết luận:** không có đơn nào có ngày mua nằm ngoài khoảng từ 01/2017 đến 08/2018.

### 5.6. Biến đặc trưng thời gian

Đếm giá trị thiếu của t_days và month.

In [ ]:
print('thieu t_days, month:', df[['t_days', 'month']].isna().sum().sum())

thieu t_days, month: 0


> **Kết luận:** t_days và month đầy đủ, số giá trị thiếu bằng 0.

### 5.7. Mã bang hợp lệ

Đếm số dòng có seller_state hoặc customer_state không thuộc 27 bang của Brazil.

In [ ]:
UF = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT', 'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS', 'RO', 'RR', 'SC', 'SP', 'SE', 'TO']

print('bang khong hop le:', (~df['seller_state'].isin(UF) | ~df['customer_state'].isin(UF)).sum())

bang khong hop le: 0


> **Kết luận:** mã bang của người bán và khách hàng đều thuộc 27 bang của Brazil, số dòng vi phạm bằng 0.

### 5.8. Outlier thống kê trên dữ liệu cuối

Tính số dòng và tỷ lệ vượt ngưỡng IQR của freight_value, price, weight_g và distance_km trên bảng cuối.

**Cách làm:**
- Ngưỡng trên của mỗi cột bằng Q3 + 1.5 × (Q3 − Q1), trong đó Q1 và Q3 là phân vị 25% và 75%.
- Đếm số dòng có giá trị lớn hơn ngưỡng và quy ra tỷ lệ phần trăm.

**Lưu ý:**
- Chỉ ghi nhận số liệu, không loại dòng nào ở bước làm sạch.
- Làm ở bước này vì distance_km chỉ có sau 4.6.

In [ ]:
cols = ['freight_value', 'price', 'weight_g', 'distance_km']
q1 = df[cols].quantile(0.25)
q3 = df[cols].quantile(0.75)
nguong = q3 + 1.5 * (q3 - q1)
so_dong = (df[cols] > nguong).sum()

print('nguong tren IQR:')
print(nguong)
print()
print('so dong vuot nguong:')
print(so_dong)
print()
print('ty le (%):')
print((so_dong / len(df) * 100).round(2))

nguong tren IQR:
freight_value      33.280000
price             277.400000
weight_g         4050.000000
distance_km      1703.484894
dtype: float64

so dong vuot nguong:
freight_value    11167
price             8018
weight_g         15256
distance_km       8270
dtype: int64

ty le (%):
freight_value    10.31
price             7.40
weight_g         14.08
distance_km       7.63
dtype: float64


> **Kết luận:** weight_g có tỷ lệ vượt ngưỡng IQR cao nhất (14.08%), các dòng này chỉ được ghi nhận và không bị loại.
>
> | Biến | Ngưỡng trên | Số dòng vượt ngưỡng | Tỷ lệ |
> |---|---|---|---|
> | freight_value | 33.28 | 11,167 | 10.31% |
> | price | 277.40 | 8,018 | 7.40% |
> | weight_g | 4,050.00 | 15,256 | 14.08% |
> | distance_km | 1,703.48 | 8,270 | 7.63% |

## 6. Tự đánh giá và kết luận

**Mục đích:** nêu các giới hạn của dữ liệu cuối, liệt kê đầu ra và tổng kết quy trình làm sạch.

### 6.1. Giới hạn phạm vi

Đếm các dòng liên quan đến hai giới hạn của distance_km: khoảng cách bằng 0 và tọa độ được điền theo bang.

In [ ]:
print('distance_km = 0       :', (df['distance_km'] == 0).sum())
print('ty le distance = 0 (%):', round((df['distance_km'] == 0).mean() * 100, 3))
print('cung zip prefix       :', (df['seller_zip'] == df['customer_zip']).sum())
print('geo_imputed = 1       :', df['geo_imputed'].sum())
print('ty le geo_imputed (%) :', round(df['geo_imputed'].mean() * 100, 3))

distance_km = 0       : 26
ty le distance = 0 (%): 0.024
cung zip prefix       : 26
geo_imputed = 1       : 531
ty le geo_imputed (%) : 0.49


> **Kết luận:** dữ liệu cuối có bốn giới hạn cần ghi nhận khi dùng cho mô hình.
>
> - freight_value = 0: 379 dòng được giữ lại ở 3.8, việc loại hay không được quyết định sau khi khám phá dữ liệu.
> - distance_km là đường chim bay giữa tâm hai vùng bưu chính, không phải quãng đường vận chuyển thực. Có 26 dòng (0.024%) có distance_km = 0, đúng bằng 26 dòng cùng zip prefix.
> - 531 dòng (0.490%) có tọa độ được điền theo tâm bang (geo_imputed = 1), nên khoảng cách của các dòng này kém chính xác hơn.
> - Dữ liệu không có thông tin hãng vận chuyển, giá nhiên liệu và phương thức giao.

### 6.2. Đầu ra

Xuất bảng cuối sau 4.7 ra tệp olist_freight_clean.csv, rồi đọc lại tệp để so kích thước, số ô thiếu và số dòng trùng grain với df.

In [ ]:
output_final = output_dir + 'olist_freight_clean.csv'

df.to_csv(output_final, index=False, encoding='utf-8-sig')

check = pd.read_csv(output_final, parse_dates=['order_purchase_timestamp'])
print('df hien tai:', df.shape)
print('file da luu:', check.shape)
print('tong missing:', check.isna().sum().sum())
print('trung grain:', check[GRAIN].duplicated().sum())

df hien tai: (108336, 28)
file da luu: (108336, 28)
tong missing: 0
trung grain: 0


> **Kết luận:** tệp olist_freight_clean.csv khớp với df, không còn ô thiếu và không trùng grain.
>
> | Tệp | Nội dung | Kích thước |
> |---|---|---|
> | olist_merged_clean.csv | Dataset sau làm sạch (3.11) | 108,336 dòng × 25 cột |
> | audit_trail.csv | Lịch sử 5 bước loại dòng (3.10) | 5 dòng |
> | olist_freight_clean.csv | Bảng cuối sau chuyển đổi và đổi tên (4.7) | 108,336 dòng × 28 cột |

### 6.3. Kết luận

Bảng cuối gồm 108,336 dòng và 28 cột, đã được làm sạch và chuyển đổi, sẵn sàng cho bước khám phá dữ liệu và xây dựng mô hình dự đoán freight_value.

- Các dòng thiếu thông tin không thể suy luận hoặc sai miền được loại bỏ, tổng cộng 4,314 dòng (3.83%), từ 112,650 xuống 108,336.
- Tọa độ thiếu được điền theo tâm bang thay vì xóa, ảnh hưởng 531 dòng (0.490%).
- Outlier thống kê và 379 dòng freight_value = 0 được giữ lại, việc xử lý được quyết định sau khi khám phá dữ liệu.
- Grain là một sản phẩm trong một đơn hàng, xác định bởi cặp (order_id, order_item_id).